## Construct pydantic model from text input

In [3]:
from dotenv import load_dotenv, find_dotenv
import os

# Detta letar uppåt tills den hittar din .env-fil i projektets rot
load_dotenv(find_dotenv())

print(f"Nu fungerar det: {os.getenv('GOOGLE_API_KEY') is not None}")

Nu fungerar det: True


In [6]:
from pydantic_ai import Agent 

agent = Agent(model="google-gla:gemini-2.5-flash")

result = await agent.run("Give me an IT employee working in Sweden, keep it short, only name, worktitle and location and country")
result

AgentRunResult(output='**Name:** Emma Andersson\n**Work Title:** IT Support Specialist\n**Location:** Malmö, Sweden')

In [8]:
print(result.output)

**Name:** Emma Andersson
**Work Title:** IT Support Specialist
**Location:** Malmö, Sweden


In [9]:
from pydantic import BaseModel, Field


class EmployeeModel(BaseModel):

    name: str
    age: int
    salary: int =Field(gt=30_000, lt=50_000)
    position: str

result = await agent.run(
    "Give me an IT employee working in sweden", output_type=EmployeeModel
)

result

AgentRunResult(output=EmployeeModel(name='Bjorn Borg', age=45, salary=45000, position='Software Engineer'))

In [11]:
employee = result.output
employee

EmployeeModel(name='Bjorn Borg', age=45, salary=45000, position='Software Engineer')

In [13]:
employee.name, employee.age, employee.position

('Bjorn Borg', 45, 'Software Engineer')

In [14]:
employee.model_dump()

{'name': 'Bjorn Borg',
 'age': 45,
 'salary': 45000,
 'position': 'Software Engineer'}

In [16]:
print(employee.model_dump_json(indent=2))

{
  "name": "Bjorn Borg",
  "age": 45,
  "salary": 45000,
  "position": "Software Engineer"
}


several employees or a list of employees

In [18]:
result = await agent.run(
    """Give me ten employess in AI and data engineering fields,
    roles can vary, but salary must be between 30000 and 50000
    """,
    output_type=list[EmployeeModel]
)

employees = result.output
employees

[EmployeeModel(name='Alice Smith', age=30, salary=45000, position='AI Engineer'),
 EmployeeModel(name='Bob Johnson', age=35, salary=48000, position='Data Engineer'),
 EmployeeModel(name='Charlie Brown', age=28, salary=40000, position='Machine Learning Engineer'),
 EmployeeModel(name='Diana Prince', age=40, salary=49999, position='Lead Data Scientist'),
 EmployeeModel(name='Eve Adams', age=32, salary=42000, position='AI Researcher'),
 EmployeeModel(name='Frank White', age=38, salary=47000, position='Big Data Engineer'),
 EmployeeModel(name='Grace Lee', age=29, salary=38000, position='Junior AI Developer'),
 EmployeeModel(name='Harry Kim', age=45, salary=49000, position='Senior Data Analyst'),
 EmployeeModel(name='Ivy Chen', age=31, salary=43000, position='Data Science Consultant'),
 EmployeeModel(name='Jack Evans', age=33, salary=46000, position='Cloud AI Engineer')]

In [19]:
len(employees)

10

In [20]:
for employee in employees:
    print(f"{employee.name =} and {employee.salary =}")

employee.name ='Alice Smith' and employee.salary =45000
employee.name ='Bob Johnson' and employee.salary =48000
employee.name ='Charlie Brown' and employee.salary =40000
employee.name ='Diana Prince' and employee.salary =49999
employee.name ='Eve Adams' and employee.salary =42000
employee.name ='Frank White' and employee.salary =47000
employee.name ='Grace Lee' and employee.salary =38000
employee.name ='Harry Kim' and employee.salary =49000
employee.name ='Ivy Chen' and employee.salary =43000
employee.name ='Jack Evans' and employee.salary =46000


## CV or resume model - a more complex and nested model

In [22]:
class ExperienceModel(BaseModel):
    titel: str
    company: str
    description: str
    start_year: int
    end_year: int


class EducationModel(BaseModel):
    titel: str
    education_area: str
    school: str
    description: str
    start_year: int
    end_year: int

class Cvmodel(BaseModel):
    name: str
    age: int
    experiences: list[ExperienceModel]
    education: list[EducationModel]

result = await agent.run(
    "Create a swedish person applying for a data engineering position",
    output_type=Cvmodel
)

resume = result.output

resume

Cvmodel(name='Astrid Lindgren', age=30, experiences=[ExperienceModel(titel='Data Engineer', company='Swedbank', description='Developed and maintained data pipelines using AWS Glue and Apache Spark. Implemented data quality checks and optimized ETL processes.', start_year=2020, end_year=2023), ExperienceModel(titel='Junior Data Engineer', company='Ericsson', description='Assisted in building and deploying data solutions. Monitored data pipelines and resolved issues.', start_year=2018, end_year=2020)], education=[EducationModel(titel='Master of Science in Computer Science', education_area='Data Engineering', school='KTH Royal Institute of Technology', description='Specialized in distributed systems and big data technologies.', start_year=2016, end_year=2018), EducationModel(titel='Bachelor of Science in Software Engineering', education_area='Software Development', school='Chalmers University of Technology', description='Focused on object-oriented programming and database design.', start_

In [ ]:
resume.name, resume.age, 

('Astrid Lindgren', 30)

In [26]:
resume.experiences[0].titel

'Data Engineer'

## Optional postprocessing -> Load into duckdb and unnest

In [27]:
import dlt

pipeline = dlt.pipeline(
    pipeline_name="resume_json_duckdb",
    destination=dlt.destinations.duckdb("cv.duckdb"),
    dataset_name="staging"
)

info = pipeline.run(data=[resume.model_dump()], loader_file_format="jsonl", table_name="cv_entries")
print(info)


Pipeline resume_json_duckdb load step completed in 0.46 seconds
1 load package(s) were loaded to destination duckdb and into dataset staging
The duckdb destination used duckdb:///c:\Users\Casper\Documents\ai_engineering_casper_zanichelli\ai_engineering_casper_zanichelli\video-alongs\cv.duckdb location to store data
Load package 1768178557.6980565 is LOADED and contains no failed jobs


In [29]:
import duckdb

with duckdb.connect("cv.duckdb") as conn:
    desc = conn.sql("desc").df()
    cv_entries =  conn.sql("from staging.cv_entries").df()
    educations = conn.sql("from staging.cv_entries__education").df()
    experiences = conn.sql("from staging.cv_entries__experiences").df()
desc

,database,schema,name,column_names,column_types,temporary
0,cv,staging,_dlt_loads,"[load_id, schema_name, status, inserted_at, sc...","[VARCHAR, VARCHAR, BIGINT, TIMESTAMP WITH TIME...",False
1,cv,staging,_dlt_pipeline_state,"[version, engine_version, pipeline_name, state...","[BIGINT, BIGINT, VARCHAR, VARCHAR, TIMESTAMP W...",False
2,cv,staging,_dlt_version,"[version, engine_version, inserted_at, schema_...","[BIGINT, BIGINT, TIMESTAMP WITH TIME ZONE, VAR...",False
3,cv,staging,cv_entries,"[name, age, _dlt_load_id, _dlt_id]","[VARCHAR, BIGINT, VARCHAR, VARCHAR]",False
4,cv,staging,cv_entries__education,"[titel, education_area, school, description, s...","[VARCHAR, VARCHAR, VARCHAR, VARCHAR, BIGINT, B...",False
5,cv,staging,cv_entries__experiences,"[titel, company, description, start_year, end_...","[VARCHAR, VARCHAR, VARCHAR, BIGINT, BIGINT, VA...",False


In [30]:
cv_entries

,name,age,_dlt_load_id,_dlt_id
0,Astrid Lindgren,30,1768178557.6980565,uF8mzNInR50kIA


In [31]:
educations

,titel,education_area,school,description,start_year,end_year,_dlt_parent_id,_dlt_list_idx,_dlt_id
0,Master of Science in Computer Science,Data Engineering,KTH Royal Institute of Technology,Specialized in distributed systems and big dat...,2016,2018,uF8mzNInR50kIA,0,1B3GQkSmsbVdRQ
1,Bachelor of Science in Software Engineering,Software Development,Chalmers University of Technology,Focused on object-oriented programming and dat...,2013,2016,uF8mzNInR50kIA,1,5zjtdpGUQB13KA


In [32]:
experiences

,titel,company,description,start_year,end_year,_dlt_parent_id,_dlt_list_idx,_dlt_id
0,Data Engineer,Swedbank,Developed and maintained data pipelines using ...,2020,2023,uF8mzNInR50kIA,0,B/vCoZP7B2zGUw
1,Junior Data Engineer,Ericsson,Assisted in building and deploying data soluti...,2018,2020,uF8mzNInR50kIA,1,PToJCFfgXZho/g


In [37]:
duckdb.sql("""
    SELECT
        cv.name,
        cv.age,
        ex.company,
        ex.description AS experience_description,
        ex.start_year AS experience_start_year,
        ex.end_year AS experience_end_year,
        e.titel,
        e.education_area,
        e.school,
        e.start_year AS education_start_year,
        e.end_year AS education_end_year       
    FROM cv_entries cv
    LEFT JOIN educations e ON cv._dlt_id = e._dlt_parent_id
    LEFT JOIN experiences ex ON cv._dlt_id = ex._dlt_parent_id


""").df()

,name,age,company,experience_description,experience_start_year,experience_end_year,titel,education_area,school,education_start_year,education_end_year
0,Astrid Lindgren,30,Ericsson,Assisted in building and deploying data soluti...,2018,2020,Master of Science in Computer Science,Data Engineering,KTH Royal Institute of Technology,2016,2018
1,Astrid Lindgren,30,Ericsson,Assisted in building and deploying data soluti...,2018,2020,Bachelor of Science in Software Engineering,Software Development,Chalmers University of Technology,2013,2016
2,Astrid Lindgren,30,Swedbank,Developed and maintained data pipelines using ...,2020,2023,Master of Science in Computer Science,Data Engineering,KTH Royal Institute of Technology,2016,2018
3,Astrid Lindgren,30,Swedbank,Developed and maintained data pipelines using ...,2020,2023,Bachelor of Science in Software Engineering,Software Development,Chalmers University of Technology,2013,2016
